# R2-01 · Datos: traer, cruzar y limpiar — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional y de **transferencia**: el mismo oficio sobre otro dato del Estado. Cruza los conteos
de **matrícula 2025 por RBD** con el **directorio oficial** de establecimientos (MINEDUC):
verificar que la llave sea única, comparar **inner vs left**, cazar las **filas huérfanas** reales
y decidir qué hacer con ellas sin perder estudiantes del total.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, os, urllib.request
for archivo in ["matricula_rbd.csv", "directorio_establecimientos.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)
mat = pd.read_csv("matricula_rbd.csv")                  # conteos de estudiantes por RBD
directorio = pd.read_csv("directorio_establecimientos.csv")  # quién es cada RBD
print("matrícula:", mat.shape, "| directorio:", directorio.shape)

## 1. Antes de cruzar: ¿la llave es única? Y después: inner vs left

Si el RBD se repitiera en el directorio, el cruce **multiplicaría** filas. Verifica la unicidad con `.is_unique` y luego compara cuántas filas sobreviven con `inner` y con `left`.

### ✍️ Ejercicio 1 — Verifica la llave y compara inner vs left

In [ ]:
llave_unica = bool(mat["rbd"].is_unique and directorio["rbd"].is_unique)
inner = mat.merge(directorio, on="rbd", how="inner")
left = mat.merge(directorio, on="rbd", how="left")
print("llave única en ambas:", llave_unica, "| inner:", len(inner), "| left:", len(left))

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert llave_unica == bool(mat["rbd"].is_unique and directorio["rbd"].is_unique)
    assert len(left) == len(mat)
    assert len(inner) < len(left)
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "mat['rbd'].is_unique and directorio['rbd'].is_unique; mat.merge(directorio, on='rbd', how='inner'/'left').")
    print("   Detalle:", e)

## 2. Cazar las huérfanas y medir su peso

Las filas que el `inner` botó en silencio son las que en el `left` quedaron sin región. ¿Cuántas son y **cuántos estudiantes** representan? Ese número decide si importan.

### ✍️ Ejercicio 2 — Cuenta huérfanas y sus estudiantes

In [ ]:
huerfanas = left[left["region"].isna()]
n_huerfanas = len(huerfanas)
est_huerfanos = int(huerfanas["mat_total"].sum())
print("huérfanas:", n_huerfanas, "| estudiantes en ellas:", est_huerfanos)
huerfanas

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert n_huerfanas == len(mat) - len(inner)
    assert est_huerfanos == int(left.loc[left["region"].isna(), "mat_total"].sum())
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "left[left['region'].isna()]; len(...) y ['mat_total'].sum().")
    print("   Detalle:", e)

## 3. Imputar con una categoría explícita (en vez de botar)

Una decisión defendible: no botar las huérfanas, sino **etiquetarlas** como `"Sin directorio"` con `fillna`. Así ningún estudiante desaparece del total y el informe muestra la categoría a la vista.

### ✍️ Ejercicio 3 — Etiqueta las huérfanas y resume por dependencia

In [ ]:
cruce = left.copy()
cruce["dependencia"] = cruce["dependencia"].fillna("Sin directorio")
est_dep = cruce.groupby("dependencia")["mat_total"].sum().sort_values(ascending=False)
print(est_dep)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert "Sin directorio" in est_dep.index
    assert int(est_dep.sum()) == int(mat["mat_total"].sum())
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "cruce['dependencia'].fillna('Sin directorio'); luego groupby('dependencia')['mat_total'].sum().sort_values(ascending=False).")
    print("   Detalle:", e)

---

*Fin de la profundización de R2-01.*